In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_7.py ──
"""
Shared infrastructure for Exercise 7 — Transfer Learning.

Contains: CIFAR-10 data loading, feature visualisation helpers,
ExperimentTracker/ModelRegistry setup, training harness, and the serving
helpers (flat-row ONNX adapter, request records, ONNX artifact attach).
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.engines.model_registry import LocalFileArtifactStore
from kailash_ml.types import MetricSpec


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex7_transfer_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The ModelRegistry stores artifact files (model.pkl, model.onnx) in an
# ArtifactStore. We construct the store explicitly so Part 5 can attach an
# ONNX serving artifact to a registered version (see attach_onnx_artifact).
ARTIFACT_STORE = LocalFileArtifactStore(".kailash_ml/artifacts")

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — CIFAR-10 (full 50K, resized for ResNet-18)
# ════════════════════════════════════════════════════════════════════════
# ResNet-18 was designed for 224x224 ImageNet images. CIFAR-10 is 32x32.
# We resize to 96x96: ResNet's strided convolutions shrink spatial dims
# by 32x, so 32x32 would collapse to 1x1 before final pooling. 96x96
# gives a 3x3 final feature map — enough spatial information to learn.

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SIZE = 96
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Standard transforms: ImageNet normalisation + resize for ResNet
train_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.RandomHorizontalFlip(),
        T.RandomCrop(INPUT_SIZE, padding=8),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
val_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def load_cifar10() -> tuple[
    torchvision.datasets.CIFAR10,
    torchvision.datasets.CIFAR10,
    DataLoader,
    DataLoader,
]:
    """Load CIFAR-10 with ImageNet-style transforms.

    Returns:
        train_set, val_set, train_loader, val_loader
    """
    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=train_transform,
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=val_transform,
    )

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, num_workers=0)

    print(
        f"CIFAR-10 (full): train={len(train_set)}, val={len(val_set)}, "
        f"classes={N_CLASSES}"
    )
    print(f"  Input size: {INPUT_SIZE}x{INPUT_SIZE} (resized for ResNet-18)")
    print(f"  Classes: {CLASS_NAMES}")

    return train_set, val_set, train_loader, val_loader


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_transfer.db"
    registry_db = "sqlite:///mlfp05_transfer_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn, artifact_store=ARTIFACT_STORE)
    return conn, tracker, "m5_transfer_learning", registry, True


def init_engines() -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# TRAINING HARNESS — shared by all technique files
# ════════════════════════════════════════════════════════════════════════


async def _train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Train a model and log everything to ExperimentTracker.

    Returns:
        train_losses, val_accs, train_accs (per-epoch)
    """
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"\n-- {name} --  trainable params: {n_trainable:,} / {n_total:,}")

    opt = torch.optim.Adam(params, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    train_losses: list[float] = []
    val_accs: list[float] = []
    train_accs: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "trainable_params": str(n_trainable),
                "total_params": str(n_total),
                "epochs": str(epochs),
                "lr": str(lr),
                "batch_size": str(tr_loader.batch_size),
                "dataset_size": str(len(tr_loader.dataset)),
            }
        )

        for epoch in range(epochs):
            # -- Training --
            model.train()
            batch_losses = []
            correct = total = 0
            for xb, yb in tr_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
                correct += int((logits.argmax(dim=-1) == yb).sum().item())
                total += int(yb.size(0))
            train_losses.append(float(np.mean(batch_losses)))
            train_accs.append(correct / total)
            scheduler.step()

            # -- Validation --
            model.eval()
            correct = total = 0
            with torch.no_grad():
                for xb, yb in vl_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    preds = model(xb).argmax(dim=-1)
                    correct += int((preds == yb).sum().item())
                    total += int(yb.size(0))
            val_accs.append(correct / total)

            await run.log_metrics(
                {
                    "train_loss": train_losses[-1],
                    "train_acc": train_accs[-1],
                    "val_acc": val_accs[-1],
                },
                step=epoch + 1,
            )

            print(
                f"  epoch {epoch + 1}/{epochs}  "
                f"loss={train_losses[-1]:.4f}  "
                f"train_acc={train_accs[-1]:.3f}  "
                f"val_acc={val_accs[-1]:.3f}"
            )

        await run.log_metrics(
            {
                "final_val_acc": val_accs[-1],
                "best_val_acc": max(val_accs),
                "final_train_loss": train_losses[-1],
            }
        )

    return train_losses, val_accs, train_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        _train_model_async(
            model,
            name,
            tracker,
            exp_name,
            tr_loader,
            vl_loader,
            epochs,
            lr,
        )
    )


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Register a trained model in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="val_acc", value=val_acc),
            MetricSpec(name="final_loss", value=final_loss),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={val_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, val_acc, final_loss))


# ════════════════════════════════════════════════════════════════════════
# SERVING HELPERS — ONNX export + InferenceServer
# ════════════════════════════════════════════════════════════════════════
IMAGE_SHAPE = (3, INPUT_SIZE, INPUT_SIZE)
N_PIXELS = 3 * INPUT_SIZE * INPUT_SIZE


class FlatImageAdapter(nn.Module):
    """Wrap an image classifier so it accepts flat pixel rows.

    InferenceServer's ONNX runtime turns each request record into ONE row of
    a 2-D float array, so the exported graph must take ``(batch, N_PIXELS)``.
    This adapter reshapes rows back to ``(batch, 3, 96, 96)`` (already
    resized and ImageNet-normalised) before calling the wrapped network.
    ``predict(X)`` is the method ``OnnxBridge.validate`` calls to get the
    native (PyTorch) outputs for comparison.
    """

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x.reshape(-1, *IMAGE_SHAPE))

    def predict(self, X) -> np.ndarray:
        model_device = next(self.model.parameters()).device
        with torch.no_grad():
            x = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=model_device)
            return self.forward(x).cpu().numpy()


def images_to_records(images: torch.Tensor) -> list[dict[str, float]]:
    """Turn a batch of preprocessed images into InferenceServer records.

    One record per image, one key per pixel. Keys are zero-padded so the
    record order is the pixel order the ONNX graph expects.
    """
    rows = images.reshape(len(images), -1).cpu().tolist()
    return [{f"px{j:05d}": float(v) for j, v in enumerate(row)} for row in rows]


def attach_onnx_artifact(name: str, version: int, onnx_path: Path) -> None:
    """Store an exported ONNX file as ``model.onnx`` for a registered version.

    ``ModelRegistry.register_model`` saves the training artifact as
    ``model.pkl``; ``InferenceServer.from_registry(..., runtime="onnx")``
    loads ``model.onnx`` for the same name + version from this ArtifactStore.

    OnnxBridge writes large weights to a sibling ``<file>.onnx.data`` file
    (ONNX external data). Storing only the ``.onnx`` bytes would leave the
    served model without its weights ("external data path does not exist"),
    so the model is loaded with its external data and stored as ONE
    self-contained protobuf.
    """
    import onnx

    model_proto = onnx.load(str(onnx_path))  # pulls in any .onnx.data weights
    asyncio.run(
        ARTIFACT_STORE.save(name, version, model_proto.SerializeToString(), "model.onnx")
    )


# ════════════════════════════════════════════════════════════════════════
# FEATURE EXTRACTION & VISUALISATION HELPERS
# ════════════════════════════════════════════════════════════════════════


def extract_features(
    model: nn.Module,
    loader: DataLoader,
    max_samples: int = 2000,
) -> tuple[np.ndarray, np.ndarray]:
    """Extract features from the penultimate layer (before fc head).

    Works for both ResNet (avgpool hook) and Sequential models.
    """
    model.eval()
    hook_features: list[torch.Tensor] = []
    labels: list[np.ndarray] = []

    def hook_fn(module, inp, out):
        del module, inp  # PyTorch hook protocol args; not used here
        hook_features.append(out.flatten(1).detach().cpu())

    # ResNet: hook into avgpool; Sequential: second-to-last layer
    if hasattr(model, "avgpool"):
        avgpool = model.avgpool
        assert isinstance(
            avgpool, nn.Module
        ), f"expected nn.Module for avgpool, got {type(avgpool).__name__}"
        handle = avgpool.register_forward_hook(hook_fn)
    else:
        assert isinstance(
            model, nn.Sequential
        ), f"hook fallback requires nn.Sequential, got {type(model).__name__}"
        handle = model[-3].register_forward_hook(hook_fn)

    with torch.no_grad():
        collected = 0
        for xb, yb in loader:
            if collected >= max_samples:
                break
            xb = xb.to(device)
            model(xb)
            labels.append(yb.numpy())
            collected += len(yb)

    handle.remove()
    features_np = torch.cat(hook_features, dim=0).numpy()[:max_samples]
    labels_np = np.concatenate(labels)[:max_samples]
    return features_np, labels_np


def compute_tsne(features: np.ndarray, perplexity: int = 30) -> np.ndarray:
    """Run t-SNE dimensionality reduction to 2D."""
    # sklearn 1.5+ renamed n_iter → max_iter in TSNE.__init__
    tsne = TSNE(n_components=2, perplexity=perplexity, max_iter=500, random_state=42)
    return tsne.fit_transform(features)


def cluster_quality(coords: np.ndarray, labels: np.ndarray) -> float:
    """Cluster quality: ratio of intra-class to inter-class distance (lower = better)."""
    intra = []
    centroids = []
    for c in range(N_CLASSES):
        mask = labels == c
        if mask.sum() < 2:
            continue
        pts = coords[mask]
        centroid = pts.mean(axis=0)
        centroids.append(centroid)
        intra.append(np.mean(np.linalg.norm(pts - centroid, axis=1)))
    centroids_arr = np.array(centroids)
    inter = np.mean(
        [
            np.linalg.norm(centroids_arr[i] - centroids_arr[j])
            for i in range(len(centroids_arr))
            for j in range(i + 1, len(centroids_arr))
        ]
    )
    avg_intra = np.mean(intra)
    return float(avg_intra / inter) if inter > 0 else float("inf")


def plot_tsne(
    coords: np.ndarray,
    labels: np.ndarray,
    title: str,
    output_path: str | Path,
) -> None:
    """Create and save a t-SNE scatter plot coloured by class."""
    fig = go.Figure()
    for c in range(N_CLASSES):
        mask = labels == c
        fig.add_trace(
            go.Scatter(
                x=coords[mask, 0],
                y=coords[mask, 1],
                mode="markers",
                name=CLASS_NAMES[c],
                marker=dict(size=4, opacity=0.6),
            )
        )
    fig.update_layout(
        title=title,
        xaxis_title="t-SNE 1",
        yaxis_title="t-SNE 2",
        template="plotly_white",
    )
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance.

    ModelVisualizer carries a kailash-ml P2 experimental notice (a
    UserWarning at construction); exercises run under
    warnings-as-errors, so the notice is acknowledged narrowly at
    this construction site.
    """
    import warnings as _warnings

    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(metrics=metrics, x_label="Epoch", y_label="Value")
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def classifier_diag_loss(model: nn.Module, batch) -> torch.Tensor:
    """Cross-entropy loss for ``run_diagnostic_checkpoint`` on an image classifier.

    The checkpoint passes each raw ``(images, labels)`` batch from the
    DataLoader; it does not move tensors, so we move them to the model's
    device here.
    """
    xb, yb = batch
    model_device = next(model.parameters()).device
    return F.cross_entropy(model(xb.to(model_device)), yb.to(model_device))


def count_params(model: nn.Module, trainable_only: bool = False) -> int:
    """Count parameters in a model."""
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 7, Part 5: Production Deployment (ONNX + Serving)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this section, you will be able to:
#   - Export a fine-tuned transfer model to ONNX format with OnnxBridge
#   - Understand why ONNX matters for production (portability, speed)
#   - Serve predictions with kailash-ml InferenceServer from the registry
#   - Compare PyTorch and ONNX Runtime latency and throughput
#   - Deploy the best model for a medical imaging use case with
#     concrete latency benchmarks and serving cost analysis
#
# PREREQUISITES: Parts 1-4 (all transfer learning techniques).
# ESTIMATED TIME: ~20 min
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import copy
import time
from pathlib import Path

import numpy as np
import onnxruntime as ort
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision

from kailash_ml import InferenceServer, OnnxBridge
from kailash_ml.diagnostics import run_diagnostic_checkpoint



## THEORY — From Experiment to Production


In [ ]:
# Training a model is only half the job. The other half is deploying it
# so that real users can get predictions. This requires:
#
# 1. MODEL EXPORT — Convert from PyTorch (Python-specific) to a portable
#    format. ONNX (Open Neural Network Exchange) is the industry standard:
#      - Runs on any ONNX runtime (CPU, GPU, mobile, edge devices)
#      - Often faster on CPU than eager PyTorch (the runtime fuses and
#        optimises the graph) — but measure it: Task 4 does
#      - Language-agnostic: serve from C++, Java, C#, not just Python
#      - Hardware-agnostic: same model on NVIDIA, AMD, Intel, Apple
#
# 2. MODEL SERVING — Wrap the model in an inference server that handles:
#      - Batch prediction (process multiple inputs efficiently)
#      - Caching (don't recompute identical predictions)
#      - Monitoring (track latency, throughput, error rates)
#      - Version management (roll back to a previous model version)
#
# 3. LATENCY BUDGETS — Production systems have strict latency
#    requirements. A medical imaging system might need:
#      - < 500ms per image for interactive use (clinician waiting)
#      - < 100ms per image for batch processing (overnight screening)
#      - < 50ms per image for real-time video analysis
#
# kailash-ml's OnnxBridge exports the model, ModelRegistry versions it,
# and InferenceServer loads the registered ONNX artifact and serves it —
# the full pipeline from experiment to production.



In [ ]:
print("\n" + "=" * 70)
print("  PART 5: Production Deployment (ONNX + InferenceServer)")
print("=" * 70)



## TASK 1 — Load data, engines, and train the production model


Build a ResNet-18 with frozen backbone and fresh classifier head.


In [ ]:
train_set, val_set, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()

# One name for the whole lifecycle: register, attach ONNX, serve.
PROD_MODEL_NAME = "production_resnet18_transfer"


def build_transfer_resnet(
    n_classes: int = N_CLASSES,
    freeze_backbone: bool = True,
) -> nn.Module:
    # No fallback to random weights: if the ImageNet download fails this
    # raises, because a random frozen backbone would make every "transfer"
    # number below meaningless.
    # TODO: Load the ImageNet ResNet-18, optionally freeze it, replace the head
    # Hint: same builder as Parts 2-4 — torchvision's ResNet18_Weights enum,
    #   requires_grad on every parameter, and a fresh nn.Linear for model.fc
    weights = ____  # TODO: the ImageNet-1K weights enum
    model = ____  # TODO: build resnet18 with those weights

    if freeze_backbone:
        for p in model.parameters():
            ____  # TODO: freeze this parameter

    in_features = model.fc.in_features
    model.fc = ____  # TODO: new classifier head for n_classes
    return model


# Train the production transfer model
print("\nTraining production transfer model...")
prod_model = build_transfer_resnet()
prod_losses, prod_accs, prod_train_accs = train_model(
    prod_model,
    "production_resnet18",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)
best_prod_acc = max(prod_accs)

# Register in ModelRegistry (model.pkl = the trained weights)
prod_version = register_model(
    registry,
    PROD_MODEL_NAME,
    prod_model,
    best_prod_acc,
    prod_losses[-1],
)



### Checkpoint 1


In [ ]:
assert best_prod_acc > 0.50, f"Production model acc {best_prod_acc:.3f} too low"
# INTERPRETATION: This is the model we're deploying. It's tracked in
# ExperimentTracker, registered in ModelRegistry, and now we'll export
# it to ONNX for portable, optimised inference.
print(f"\n  Production model val_acc: {best_prod_acc:.3f}")
print("--- Checkpoint 1 passed --- production model trained\n")



## DIAGNOSTIC CHECKPOINT — pre-export gate


In [ ]:
# Before shipping, run kailash-ml's instrumented diagnostic pass on the
# model we are about to export: a few real forward/backward passes over
# validation batches (no optimiser step) with gradient, activation and
# dead-neuron hooks attached, plus the real training-loss history.
# The checkpoint puts the model in train mode, which updates BatchNorm
# running statistics, so we diagnose a COPY and leave the model we are
# about to export untouched.
print("── Diagnostic Report (Production model — pre-export gate) ──")
# TODO: Run the diagnostic checkpoint on a COPY of prod_model over val_loader
# Hint: run_diagnostic_checkpoint(model, loader, loss_fn, title=..., ...);
#   copy.deepcopy keeps BatchNorm statistics of the real model untouched;
#   the shared classifier_diag_loss is the loss_fn
diag, findings = run_diagnostic_checkpoint(
    ____,  # TODO: a copy of the production model
    val_loader,
    ____,  # TODO: loss function
    title="Production model — pre-export gate",
    n_batches=8,
    train_losses=prod_losses,
    show=False,
)
print_prescription_pad(findings, "Production model — pre-export gate")
# HOW TO READ THE PRESCRIPTION PAD FOR THIS MODEL:
#  Gradient flow — only the new fc head is trainable; the frozen backbone
#     produces no parameter gradients by design, so judge this reading by
#     the fc layer. A CRITICAL "exploding" reading on fc means the head's
#     updates are large relative to its weights: lower the learning rate
#     before retraining. A "vanishing" reading on fc means the head
#     stopped learning.
#  Dead neurons — the ReLUs belong to the frozen ImageNet backbone. A
#     high dead fraction here means many pretrained features are silent
#     on CIFAR-10 images; that is a domain-gap signal (consider unfreezing
#     the last stage or an adapter, Part 4), not a training bug.
#  Loss trend — read from the real per-epoch training losses. A
#     non-converging trend is a reason to block the export and retrain.
#  If any reading is UNKNOWN, the library could not compute it from this
#  run; the message says why.



## TASK 2 — Export to ONNX via OnnxBridge


In [ ]:
# ONNX export traces the model's computation graph with a sample input,
# then serialises graph + weights to one portable file. OnnxBridge marks
# the batch dimension dynamic (any batch size at inference) and requests
# opset 17, a widely supported operator set; recent PyTorch exporters
# may upgrade it to 18 and print a version-conversion traceback — that
# is log noise, export_result.success is the real signal.
#
# InferenceServer's ONNX runtime turns each request record into ONE row
# of a 2-D float array, so we export the network behind FlatImageAdapter:
# it takes (batch, 3*96*96) rows of preprocessed pixels and reshapes them
# to (batch, 3, 96, 96).

print("-- Exporting to ONNX with OnnxBridge --")
prod_model.eval()
prod_model_cpu = prod_model.cpu()
# TODO: Wrap the CPU model so it accepts flat pixel rows, in eval mode
serving_adapter = ____

onnx_path = OUTPUT_DIR / "transfer_resnet18.onnx"
bridge = OnnxBridge()
# TODO: Export the adapter with OnnxBridge
# Hint: bridge.export(model, framework, output_path=..., sample_input=...) —
#   the framework string for PyTorch modules is "torch", and the sample
#   input is TWO flat rows of N_PIXELS random values (a 1-row trace fixes
#   the batch dimension at 1, and the server predicts batches of 8 below)
export_result = ____
print(
    f"  OnnxBridge.export: success={export_result.success} "
    f"status={export_result.onnx_status}"
)
assert export_result.success, f"OnnxBridge export failed: {export_result.error_message}"

onnx_size_kb = onnx_path.stat().st_size // 1024
print(f"  Exported to {onnx_path} ({onnx_size_kb} KB)")

# Numerical check: OnnxBridge.validate runs the native model (through the
# adapter's predict()) and ONNX Runtime on the same rows.
val_x, val_y = next(iter(val_loader))
check_rows = val_x[:16].reshape(16, -1).numpy().astype(np.float32)
# TODO: Compare PyTorch and ONNX Runtime outputs on check_rows
# Hint: bridge.validate(...) with a tolerance of 1e-3
validation = ____
print(
    f"  OnnxBridge.validate: valid={validation.valid} "
    f"max_diff={validation.max_diff:.2e}"
)
assert validation.valid, f"ONNX output drifted from PyTorch: {validation.notes}"

# Attach the .onnx file to the version registered in Task 1, so the
# InferenceServer can load it from the registry by name + version.
# TODO: Attach onnx_path to the registered version (same name the server uses)
____
print(f"  Attached model.onnx to {PROD_MODEL_NAME} v{prod_version.version}")



### Checkpoint 2


In [ ]:
assert onnx_path.exists(), "ONNX file should be exported"
assert onnx_path.stat().st_size > 1000, "ONNX file should not be empty"
# INTERPRETATION: The ONNX file is a self-contained model artifact.
# It contains the full computation graph and all weights. You can
# deploy this file to any server with an ONNX runtime — no Python
# or PyTorch required. This is how models go from "laptop experiment"
# to "production API serving millions of requests".
print("--- Checkpoint 2 passed --- ONNX export complete\n")



## TASK 3 — Serve predictions with InferenceServer


Load one registered ONNX model version and serve a batch request.


In [ ]:
# InferenceServer.from_registry resolves a registered model by name and
# version; start() loads its model.onnx from the registry's artifact
# store; predict() takes {"records": [...]} and returns the model output
# per record.

print("-- Serving with InferenceServer --")

N_SERVE = 8
sample_x = val_x[:N_SERVE]
sample_y = val_y[:N_SERVE]


async def serve_predictions(name: str, version: int, images: torch.Tensor) -> dict:
    # TODO: Build the server from the registry, start it, predict, stop it
    # Hint: InferenceServer.from_registry is a coroutine (await it); pass
    #   registry=, version= and the "onnx" runtime. predict() takes a dict
    #   with a "records" list — images_to_records builds it.
    server = ____
    ____  # TODO: start the server
    print(f"  InferenceServer status: {server.status}  ({name} v{version}, onnx)")
    response = ____
    ____  # TODO: stop the server
    return response


response = asyncio.run(
    serve_predictions(PROD_MODEL_NAME, prod_version.version, sample_x)
)
server_logits = np.asarray(response["predictions"], dtype=np.float32)
server_preds = ____  # TODO: predicted class per served image
server_conf = F.softmax(torch.from_numpy(server_logits), dim=-1).max(dim=-1).values

# The same images through PyTorch directly, for comparison
with torch.no_grad():
    torch_preds = prod_model_cpu(sample_x).argmax(dim=-1).numpy()

print("\n  === InferenceServer Predictions (vs direct PyTorch) ===")
print(
    f"  {'#':<4} {'True':>12} {'Served':>12} {'PyTorch':>12} "
    f"{'Confidence':>12} {'Correct':>8}"
)
print("  " + "-" * 64)
for i in range(N_SERVE):
    true_cls = CLASS_NAMES[int(sample_y[i])]
    served_cls = CLASS_NAMES[int(server_preds[i])]
    torch_cls = CLASS_NAMES[int(torch_preds[i])]
    correct = "Y" if server_preds[i] == int(sample_y[i]) else "N"
    print(
        f"  {i + 1:<4} {true_cls:>12} {served_cls:>12} {torch_cls:>12} "
        f"{float(server_conf[i]):>12.3f} {correct:>8}"
    )
n_correct = int((server_preds == sample_y.numpy()).sum())
print(f"\n  Served-sample accuracy: {n_correct}/{N_SERVE}")



### Checkpoint 3


In [ ]:
assert server_logits.shape == (N_SERVE, N_CLASSES), server_logits.shape
assert np.array_equal(
    server_preds, torch_preds
), "InferenceServer (ONNX) and PyTorch disagree on the sample classes"
# INTERPRETATION: The predictions above came back from InferenceServer
# running the registered ONNX artifact — not from PyTorch — and they
# match the PyTorch classes image for image. That agreement is the
# deployment contract: what you validated in the notebook is what the
# server returns. The server is also where production concerns live:
# request batching, monitoring (P50/P99 latency, throughput) and
# version pinning (we asked for an explicit version above).
print("\n--- Checkpoint 3 passed --- InferenceServer served the ONNX model\n")



## TASK 4 — Latency benchmarks: PyTorch vs ONNX Runtime


Return per-call latencies (ms) of the PyTorch model on `device`.


Return per-call latencies (ms) of the ONNX graph in ONNX Runtime (CPU).


In [ ]:
# Production systems need to know: how fast is this model? We benchmark
# single-image and batch latency for eager PyTorch (on the training
# device) and for ONNX Runtime on CPU (the serving runtime above).

print("-- Latency Benchmarks --")

serving_model = prod_model.to(device).eval()
ort_session = ort.InferenceSession(str(onnx_path))
ort_input = ort_session.get_inputs()[0].name
n_warmup = 5
n_bench = 50


def time_pytorch(batch: torch.Tensor) -> list[float]:
    batch = batch.to(device)
    with torch.no_grad():
        for _ in range(n_warmup):
            serving_model(batch)
        latencies = []
        for _ in range(n_bench):
            # TODO: time one forward pass in milliseconds
            # Hint: time.perf_counter() before and after; on CUDA, call
            #   torch.cuda.synchronize() first so you time the GPU work
            ____
    return latencies


def time_onnx(batch: torch.Tensor) -> list[float]:
    rows = batch.reshape(len(batch), -1).numpy().astype(np.float32)
    for _ in range(n_warmup):
        ort_session.run(None, {ort_input: rows})
    latencies = []
    for _ in range(n_bench):
        # TODO: time one ort_session.run call in milliseconds
        ____
    return latencies


single_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)
batch_input = torch.randn(32, 3, INPUT_SIZE, INPUT_SIZE)

single_latencies = time_pytorch(single_input)
batch_latencies = time_pytorch(batch_input)
onnx_single_latencies = time_onnx(single_input)
onnx_batch_latencies = time_onnx(batch_input)

# TODO: P50/P99 latencies and P50 throughput
# Hint: np.percentile; throughput = images per batch / seconds per batch
single_p50 = ____
single_p99 = ____
batch_p50 = ____
batch_p99 = ____
throughput = ____  # images/sec at P50
onnx_single_p50 = np.percentile(onnx_single_latencies, 50)
onnx_batch_p50 = np.percentile(onnx_batch_latencies, 50)
onnx_speedup = single_p50 / onnx_single_p50

print(f"\n  === Latency Benchmarks (PyTorch on {device}, ONNX Runtime on CPU) ===")
print(f"  {'Metric':<34} {'Value':>15}")
print("  " + "-" * 51)
print(f"  {'PyTorch single image P50':<34} {single_p50:>12.1f} ms")
print(f"  {'PyTorch single image P99':<34} {single_p99:>12.1f} ms")
print(f"  {'PyTorch batch (32) P50':<34} {batch_p50:>12.1f} ms")
print(f"  {'PyTorch batch (32) P99':<34} {batch_p99:>12.1f} ms")
print(f"  {'PyTorch throughput (P50)':<34} {throughput:>12.0f} img/s")
print(f"  {'ONNX Runtime single image P50':<34} {onnx_single_p50:>12.1f} ms")
print(f"  {'ONNX Runtime batch (32) P50':<34} {onnx_batch_p50:>12.1f} ms")
print(f"  {'ONNX model size':<34} {onnx_size_kb:>12,} KB")

# Visualise latency distribution
fig_latency = go.Figure()
for name, values, colour in [
    ("PyTorch single image", single_latencies, "#2196F3"),
    ("PyTorch batch (32)", batch_latencies, "#4CAF50"),
    ("ONNX Runtime single image", onnx_single_latencies, "#FF9800"),
    ("ONNX Runtime batch (32)", onnx_batch_latencies, "#9C27B0"),
]:
    fig_latency.add_trace(
        go.Histogram(x=values, name=name, marker_color=colour, opacity=0.6, nbinsx=20)
    )
fig_latency.update_layout(
    title=f"Inference Latency Distribution (PyTorch on {device}, ONNX Runtime on CPU)",
    xaxis_title="Latency (ms)",
    yaxis_title="Count",
    template="plotly_white",
    barmode="overlay",
)
latency_path = OUTPUT_DIR / "05_latency_distribution.html"
fig_latency.write_html(str(latency_path))
print(f"\n  Saved: {latency_path}")



### Checkpoint 4


In [ ]:
assert single_p50 > 0, "Should have measured single-image latency"
assert batch_p50 > 0, "Should have measured batch latency"
assert throughput > 0, "Should have positive throughput"
assert onnx_single_p50 > 0, "Should have measured ONNX Runtime latency"
# INTERPRETATION: These benchmarks tell you whether the model meets
# production latency requirements. Batch processing is cheaper per image
# because the hardware parallelises across the batch. Whether ONNX
# Runtime beats PyTorch depends on hardware: on a GPU, eager PyTorch can
# win; on CPU-only servers ONNX Runtime usually does.
print(
    f"\n  Single-image P50: PyTorch {single_p50:.1f} ms vs ONNX Runtime "
    f"{onnx_single_p50:.1f} ms (ONNX speed-up x{onnx_speedup:.2f}); "
    f"budget for interactive use: 500 ms "
    f"({'within' if single_p50 < 500 else 'OVER'} budget)"
)
print("\n--- Checkpoint 4 passed --- latency benchmarks complete\n")



## TASK 5 — Apply: Medical Imaging Production Deployment


In [ ]:
# SCENARIO (illustrative): Deploy the transfer model for the public
# dermatology clinic use case from Part 2. The volumes below are
# illustrative planning figures, not data from a real clinic:
#   - 200 patients/day, ~5 images per patient = 1,000 images/day
#   - Interactive mode: a dermatologist reviews predictions in real time
#   - Batch mode: overnight screening of new submissions

print("\n" + "=" * 70)
print("  APPLY: Medical Imaging Deployment — Dermatology Clinic (illustrative)")
print("=" * 70)

PATIENTS_PER_DAY = 200
IMAGES_PER_PATIENT = 5
DAILY_IMAGES = PATIENTS_PER_DAY * IMAGES_PER_PATIENT

# Interactive mode: dermatologist reviews in real-time
# TODO: daily processing time in minutes for both modes
# Hint: interactive = one image at a time at the single-image P50;
#   batch = DAILY_IMAGES split into batches of 32 at the batch P50
interactive_time_per_image = ____  # ms
interactive_total_ms = ____
interactive_total_min = ____

# Batch mode: overnight screening
batch_n_batches = ____
batch_total_ms = ____
batch_total_min = ____

# Cost analysis (illustrative cloud GPU price — check your provider)
GPU_HOURLY_COST = 1.20  # S$/hr, illustrative
interactive_hours = interactive_total_min / 60
batch_hours = batch_total_min / 60

print(f"\n  === Deployment Profile ===")
print(
    f"  Daily volume: {PATIENTS_PER_DAY} patients x {IMAGES_PER_PATIENT} images = {DAILY_IMAGES:,} images/day"
)
print(f"  Model: ResNet-18 transfer ({best_prod_acc:.1%} accuracy)")
print(f"  ONNX size: {onnx_size_kb} KB")
print()
print(f"  {'Mode':<20} {'Per Image':>12} {'Daily Total':>14} {'GPU Cost':>12}")
print("  " + "-" * 60)
print(
    f"  {'Interactive':<20} "
    f"{interactive_time_per_image:>10.1f}ms "
    f"{interactive_total_min:>12.1f}min "
    f"S${interactive_hours * GPU_HOURLY_COST:>9.2f}"
)
print(
    f"  {'Batch (32)':<20} "
    f"{batch_p50 / 32:>10.1f}ms "
    f"{batch_total_min:>12.1f}min "
    f"S${batch_hours * GPU_HOURLY_COST:>9.2f}"
)
print()
print(f"  DEPLOYMENT RECOMMENDATION:")
if onnx_speedup > 1.0:
    print(
        f"  1. Serve the ONNX artifact: ONNX Runtime was x{onnx_speedup:.2f} faster "
        f"than PyTorch per image on this machine"
    )
else:
    print(
        f"  1. ONNX Runtime was not faster here (x{onnx_speedup:.2f}); serve ONNX for "
        f"portability, and benchmark again on the target server"
    )
print(f"  2. Use batch mode for overnight screening ({batch_total_min:.1f} min/day)")
print(
    f"  3. Interactive mode for real-time consultation ({interactive_time_per_image:.0f}ms/image)"
)
print(
    f"  4. Monthly GPU cost (illustrative, 1 h/day minimum): "
    f"~S${30 * max(interactive_hours, 1) * GPU_HOURLY_COST:.0f}"
)
print(f"  5. Register model in ModelRegistry for version tracking and rollback")

# Model comparison summary across all parts
print(f"\n  === Exercise 7 Complete Model Comparison ===")
print(f"  {'Approach':<30} {'Val Accuracy':>14} {'Trainable':>14} {'Use When':>25}")
print("  " + "-" * 85)

n_frozen_trainable = count_params(prod_model, trainable_only=True)

print(
    f"  {'From scratch (Part 1)':<30} "
    f"{'see Part 1':>14} "
    f"{'all params':>14} "
    f"{'Abundant data, unique domain':>25}"
)
print(
    f"  {'Frozen head (Part 2/5)':<30} "
    f"{best_prod_acc:>14.1%} "
    f"{n_frozen_trainable:>14,} "
    f"{'Quick start, limited compute':>25}"
)
print(
    f"  {'Adapter (Part 4)':<30} "
    f"{'see Part 4':>14} "
    f"{'see Part 4':>14} "
    f"{'Multi-tenant, balanced':>25}"
)
print(
    f"  {'LoRA (Module 6)':<30} "
    f"{'coming in M6':>14} "
    f"{'~1-5%':>14} "
    f"{'LLMs, billions of params':>25}"
)



### Checkpoint 5


In [ ]:
assert onnx_path.exists(), "ONNX model should be deployed"
assert best_prod_acc > 0.50, "Production model should have reasonable accuracy"
# INTERPRETATION: The full pipeline — train, track, register, export,
# serve, benchmark — is what production ML looks like. Every step is
# logged and versioned: you can trace from a prediction back to the
# exact model version, training run, and dataset that produced it.
# This audit trail is essential for regulated industries like healthcare.
print("\n--- Checkpoint 5 passed --- deployment analysis complete\n")



## CLEANUP


In [ ]:
await conn.close()



## REFLECTION


PART 1 — From-Scratch Baseline:
    [x] Trained CNN from random init, quantified the data bottleneck
    [x] Visualised noisy learned filters and overlapping t-SNE clusters

  PART 2 — Transfer Learning:
    [x] Loaded pre-trained ResNet-18, froze backbone, trained classifier head
    [x] Visualised structured activations, Grad-CAM attention maps
    [x] Applied to medical imaging (a public dermatology clinic, illustrative)

  PART 3 — Data Efficiency:
    [x] Measured accuracy at 10/25/50/100% of training data
    [x] Plotted efficiency curves, identified the labelling sweet spot
    [x] Answered "how many images do we need?" for a ride-hailing platform

  PART 4 — Adapter Modules:
    [x] Built bottleneck adapters with zero-init skip connections
    [x] Compared parameter efficiency: scratch vs frozen vs adapter
    [x] Analysed multi-tenant serving savings (50 clients)

  PART 5 — Production Deployment:
    [x] Exported to ONNX with OnnxBridge ({onnx_size_kb} KB portable model)
    [x] Served predictions with InferenceServer from the ModelRegistry
    [x] Benchmarked: {single_p50:.1f}ms single, {throughput:.0f} img/s throughput
    [x] Designed a medical imaging deployment for a dermatology clinic

  ARCHITECTURE-SELECTION GUIDE (consolidated across M5):
    Images    -> CNN / ViT + transfer learning (ImageNet pre-trained)
    Text      -> Transformer + transfer learning (BERT / GPT pre-trained)
    Sequences -> LSTM / Transformer (sometimes transfer)
    Tabular   -> Gradient boosting (train from scratch, fast and reliable)

  TRANSFER LEARNING SPECTRUM:
    Frozen head -> Adapter/LoRA -> Partial fine-tune -> Full fine-tune
    (fewest params)                                   (most params)
    (fastest training)                                (highest capacity)
    (safest from forgetting)                          (risk of forgetting)

  NEXT: Exercise 8 covers Reinforcement Learning (DQN + PPO).
  Then Module 6 uses LoRA and adapters for LLM fine-tuning — the same
  concept you explored here, applied to language models with billions
  of parameters.


In [ ]:
print("\n" + "=" * 70)
print("  EXERCISE 7 COMPLETE — What You've Mastered")
print("=" * 70)
print(
)

